# EDAViz in Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danielTeniente/deaviz/blob/main/notebooks/colab_edaviz.ipynb)

Run **Get the code** first, then pick one way to open the app:

- **In the browser, no server** (section 2, recommended in Colab): the app runs inside the cell output with your DataFrame loaded, exactly like pygwalker. No server, port, proxy, link or password.
- **Inside this notebook through Colab's proxy** (section 2b): the full app (including Prophet forecasting) runs on the Colab VM and is shown through Colab's port proxy. It may stay blank in some browsers.
- **Public link** (section 3): opens the app in its own tab through localtunnel. Use it if section 2 shows a blank or stuck frame.

The first start installs the requirements (1–3 minutes).

Nothing else is needed: Node.js (for the localtunnel link) is already part of Colab.

## 1. Get the code

In [ ]:
REPO = "https://github.com/danielTeniente/deaviz.git"
BRANCH = "main"

import os
if not os.path.isdir("/content/deaviz"):
    !git clone -q --depth 1 -b {BRANCH} {REPO} /content/deaviz
%cd /content/deaviz

## 2. In the browser, no server: `edaviz.walk(df, mode="browser")`

The app's code and the DataFrame are sent to your browser, which runs EDAViz with [stlite](https://github.com/whitphx/stlite) (Streamlit on WebAssembly) right in the cell output. Nothing runs on the Colab VM and nothing is tunnelled. The first start downloads Python packages into the browser (20–60 s; a progress list shows each step).

Check that:
- the app appears with *Load data* already showing **Column Selection** for the iris columns;
- the sidebar (`>>`) opens *Visualization* and plots draw.

In this mode Prophet forecasting and the *Download Processed Data* button are unavailable, and the sidebar is a plain list. Replace `df` with your own data (sample very large frames first).

In [ ]:
import edaviz
import plotly.express as px

df = px.data.iris()
browser_app = edaviz.walk(df, mode="browser")

## 2b. Inside this notebook through Colab's proxy: `edaviz.walk(df)`

Shows EDAViz in the cell output through Colab's own port proxy, with the DataFrame already loaded. Replace `df` with your own data (for example `pd.read_csv(...)`).

Check that:
- the app appears below the cell and *Load data* already shows **Column Selection** for the iris columns;
- the sidebar (`>>`) opens *Visualization* and plots draw.

If the frame stays blank or shows *Connecting…*, Colab's proxy is not passing the app's WebSocket: run `walk_app.stop()` and use section 2 or 3.

In [ ]:
import edaviz
import plotly.express as px

df = px.data.iris()
walk_app = edaviz.walk(df)       # height=1100 for a taller frame

**If the frame is blank**, run the next cell. It shows where the problem is:

1. `server running: True` and `local health: b'ok'`: the app works on the Colab VM.
2. A small frame showing **ok**: Colab's port proxy reaches the app. If it does but the app frame above stays blank or on *Connecting…*, the proxy is blocking Streamlit's live connection (WebSocket); use section 3.
3. Nothing in the small frame: the proxy itself is not serving this port; use section 3.

To see the exact error, open the browser's developer console (right-click the frame → *Inspect* → *Console*).

In [ ]:
import urllib.request
from google.colab import output

print("server running:", walk_app.running)
print("local health:", urllib.request.urlopen(walk_app.url + "_stcore/health", timeout=5).read())
output.serve_kernel_port_as_iframe(walk_app.server.port, path=walk_app.path + "_stcore/health", height="60")

In [ ]:
walk_app.stop()                  # stop it when you're done

## 3. Public link: start EDAViz

When the cell finishes it shows a link and a password:

1. Click the link.
2. localtunnel shows a reminder page: paste the **Tunnel Password** shown below the link and click **Click to Submit**.
3. EDAViz opens. Upload a CSV or load one from GitHub on the *Load data* page.

The cell returns as soon as the app is up, so you can keep using the notebook while the app runs.

In [ ]:
import edaviz

app = edaviz.run_colab()
app

## 4. Stop the public link

Closes the public link and stops the app. Run section 3 again to start a new one (the link changes each time).

In [ ]:
app.stop()

## Troubleshooting

| Problem | What to do |
|---|---|
| Section 2 shows *Error during booting up* or stays on *Installing packages* | The browser could not download Pyodide or a package from jsDelivr (offline, or blocked by a firewall or extension). Check the connection and rerun the cell, or use section 3. |
| Section 2b frame is blank or stuck on *Connecting…* | Colab's proxy did not pass the app's WebSocket. Run `walk_app.stop()` and use section 3. |
| Link shows **503 - Tunnel Unavailable** or stops loading | localtunnel dropped the connection. Run `app.stop()`, then section 3 again. |
| Asked for a password again | Use the one printed by section 3 (it is this runtime's public IP). |
| `LaunchError: localtunnel did not report a public URL` | The localtunnel service is busy or blocked; wait a minute and run section 3 again. |
| App page says *Connection error* | The app was stopped (section 4, or the runtime restarted). Run section 3 again. |
| Reinstall the requirements | `edaviz.colab.install_requirements(force=True)` |

**Privacy:** anyone with the link (and the password) can open the app while it runs, and data you upload stays in this Colab runtime. Stop the app when you're done.